# Chapter 18 &mdash; The History of Lambda Calculus, and its Independence from Turing

**Concept 1 of the Chapter 18 decomposition:** *The History of Lambda Calculus, and its Independence from Turing's Work*

Church 1935, Turing 1936 &mdash; contemporaneous, independent, and provably equivalent.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-History-Of-Lambda/Concept-History-Of-Lambda.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Church published the $\lambda$-calculus in **1935**; Turing published his machine in
**1936**. The work was **independent** &mdash; Turing was Church's doctoral student only
afterwards &mdash; and the two formalisms were soon proved **equivalent**.

That equivalence is the empirical core of the Church&ndash;Turing thesis (Chapter 13,
Concept 3). Two people, starting from completely different intuitions &mdash; a tape and
a state of mind on one side, function abstraction and application on the other &mdash;
arrived at the same class of computable functions.

The $\lambda$-calculus also answered Hilbert's Entscheidungsproblem first, though
Turing's model made the answer easier to believe: it looked like *computing*.

Its descendants are everywhere: Lisp, ML, Haskell, and the anonymous functions in
every mainstream language.

## 2. Definitions

### The two models, computing the same function

In [ ]:
# --- Church encodings, in Python lambdas --------------------------------
# A Church numeral n is the function that applies f to x, n times.
ZERO  = lambda f: lambda x: x
SUCC  = lambda n: lambda f: lambda x: f(n(f)(x))
ADD   = lambda m: lambda n: lambda f: lambda x: m(f)(n(f)(x))
MUL   = lambda m: lambda n: lambda f: m(n(f))
EXP   = lambda m: lambda n: n(m)

def church(n):
    c = ZERO
    for _ in range(n): c = SUCC(c)
    return c

def unchurch(c):
    return c(lambda k: k + 1)(0)

# Booleans: TRUE picks its first argument, FALSE its second -- so a
# Boolean IS an if-then-else.
TRUE  = lambda t: lambda f: t
FALSE = lambda t: lambda f: f
IF    = lambda b: lambda t: lambda f: b(t)(f)
AND   = lambda p: lambda q: p(q)(p)
OR    = lambda p: lambda q: p(p)(q)
NOT   = lambda p: p(FALSE)(TRUE)

def unbool(b): return b(True)(False)

# Pairs and selectors
PAIR   = lambda a: lambda b: lambda s: s(a)(b)
FIRST  = lambda p: p(TRUE)
SECOND = lambda p: p(FALSE)

# Predecessor and zero-test, which recursion needs
ISZERO = lambda n: n(lambda _: FALSE)(TRUE)
SHIFT  = lambda p: PAIR(SECOND(p))(SUCC(SECOND(p)))
PRED   = lambda n: FIRST(n(SHIFT)(PAIR(ZERO)(ZERO)))
SUB    = lambda m: lambda n: n(PRED)(m)
LEQ    = lambda m: lambda n: ISZERO(SUB(m)(n))


def tm_style_succ(n):
    # 'write a 1 at the end of a unary tape'
    tape = '1' * n
    return len(tape + '1')

### The timeline

In [ ]:
TIMELINE = [
 (1928, "Hilbert poses the Entscheidungsproblem"),
 (1935, "Church: the lambda-calculus; undecidability of lambda-equivalence"),
 (1936, "Turing: the machine, independently; and TM = lambda"),
 (1937, "Turing becomes Church's doctoral student at Princeton"),
 (1958, "McCarthy: Lisp, the first lambda-calculus-inspired language"),
 (2011, "C++11 adds lambdas -- the idea reaches the mainstream"),
]

## 3. Tests

The timeline.

In [ ]:
for y, what in TIMELINE:
    print("  %d  %s" % (y, what))

Both models compute successor, by utterly different means.

In [ ]:
for n in range(5):
    lam = unchurch(SUCC(church(n)))
    tm  = tm_style_succ(n)
    print("  n=%d : lambda %d, tape %d" % (n, lam, tm))
    assert lam == tm == n + 1

And addition, and multiplication.

In [ ]:
for m in range(4):
    for n in range(4):
        assert unchurch(ADD(church(m))(church(n))) == m + n
        assert unchurch(MUL(church(m))(church(n))) == m * n
print("ADD and MUL verified for all m, n in 0..3")
print("  3 + 4 =", unchurch(ADD(church(3))(church(4))))
print("  3 * 4 =", unchurch(MUL(church(3))(church(4))))

**Independence matters** &mdash; it is what makes the thesis convincing.

In [ ]:
print("Church started from : functions, abstraction, application")
print("Turing started from : a person with paper, pencil and a state of mind")
print()
print("Neither borrowed from the other, and they landed in the same place.")
print("That convergence is the evidence for the Church-Turing thesis.")

The descendants.

In [ ]:
LANGS = [("Lisp, 1958",       "lambda as a primitive"),
         ("ML, 1973",         "typed lambda calculus as a language"),
         ("Haskell, 1990",    "lazy evaluation, close to the calculus"),
         ("Python, 1994",     "lambda expressions (single-expression only)"),
         ("Java 8, 2014",     "lambdas and method references"),
         ("C++11, 2011",      "lambdas with explicit capture")]
for a, b in LANGS: print("  %-18s %s" % (a, b))

## 4. Exercises


1. Why did Turing's model convince people faster than Church's?
2. Read Turing's 1936 appendix, where he proves the two equivalent. How long is it?
3. Which mainstream language has the most faithful lambda? The least?

In [ ]:
# Your work for the exercises above.